# Flight Delay Project  
## COMP4381 – Data Science and Analytics  

**Instructor:** Ahmed Sabbah  

**Raheeq Mousa:** 1220515  
**Aya Abdakareem:** 1220020  
**Zaid Mousa:** 1221833  

---

In [3]:
print("Welcome to the Flight Delay Project")

Welcome to the Flight Delay Project


In [5]:
pip install gdown

<p style="font-weight:bold;font-size:18px">RUN THE CELL BELOW ONLY IF YOU HAD NOT DOWNLOADED THE DATASET BEFORE!</p>

In [ ]:
#YOU DON'T NEED TO RUN THIS CELL EVERYTIME, ONLY IF YOU HADN'T DOWNLOADED THE DATASET
import gdown

file_id = "1BBnDY8JWrR-0faAEmZmKaDEHIZE6FreQ"
url = f"https://drive.google.com/uc?id={file_id}"
output = "data/raw/flight_data_2024.csv"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1BBnDY8JWrR-0faAEmZmKaDEHIZE6FreQ
From (redirected): https://drive.google.com/uc?id=1BBnDY8JWrR-0faAEmZmKaDEHIZE6FreQ&confirm=t&uuid=7e7a951c-ec50-4a72-aac9-d4da9de9cce1
To: C:\Users\victus\OneDrive\Attachments\Desktop\Raheeq\Smstrs\8th_smstr_RaheeqMousa\Data science\Flight-Delay-Prediction\data\raw\flight_data_2024.csv
  4%|▎         | 48.2M/1.31G [01:11<36:15, 579kB/s]

In [2]:
import pandas as pd
import numpy as np

In [275]:
df= pd.read_csv('data/raw/flight_data_2024.csv')

C:\Users\victus\AppData\Local\Temp\ipykernel_6468\1977606443.py:1: DtypeWarning: Columns (0: cancellation_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv('data/raw/flight_data_2024.csv')


First lets remove the cancelled and diverted flights from the dataframe since they don't have arrival delay<br>
and i can't classifiy them whether they are delayed or not

In [283]:
df = df[df["cancelled"] == 0]
df = df[df["diverted"] == 0]

# Sampling

create a new column weather the flight is delayed or not (0 = not delayed / 1 = delayed)

In [287]:
df['delayed']=(df['arr_delay']>=15).astype(int)

Making Stratified Sampling while keepingg aware of the distribution stay the same

In [290]:
print(f"Before a sampling, ratio of delayed flights {df['delayed'].mean()} and ratio of non delayed {(df['delayed']==0).mean()}")

total=1000000
delayed_ratio= df['delayed'].mean()

number_delayed= int(total*delayed_ratio)
number_non_delayed= total-number_delayed

delayed_flights=df[df['delayed']==1].sample(n=number_delayed,random_state=7)
non_delayed_flights=df[df['delayed']==0].sample(n=number_non_delayed,random_state=7)

sampled_flights=pd.concat([delayed_flights,non_delayed_flights]).sample(frac=1, random_state=4)
print(f"After a stratified sampling, ratio of delayed flights {number_delayed/total} and ratio of non delayed {number_non_delayed/total}")

Before a sampling, ratio of delayed flights 0.20817177575532997 and ratio of non delayed 0.79182822424467
After a stratified sampling, ratio of delayed flights 0.208171 and ratio of non delayed 0.791829


Write the sampled data to a new file

In [293]:
sampled_flights.to_csv('data/sampled/sampled_flights.csv')

# Data Collection and merging the data

In [487]:
df = pd.read_csv('data/sampled/sampled_flights.csv')

In [488]:
df=df[['year','month','day_of_month','day_of_week',
'fl_date','distance',
'origin','origin_city_name','origin_state_nm','dest', 
'dest_city_name','dest_state_nm','op_unique_carrier','op_carrier_fl_num',
'crs_dep_time','crs_arr_time','delayed']]


I fetched the airport data long and lat inorder to get the latitude and longitudde fro the origin and dest airports.

In [492]:
airports = pd.read_csv("data/raw/airports.csv") 
airport_cord = airports[["iata", "lat", "lon"]]

srop the rows that contains na values

In [495]:
airport_cord = airport_cord.dropna()

check the correctness of the lat and lng

In [498]:
airport_cord[(airport_cord['lat'] < -90) | (airport_cord['lat'] > 90)]

,iata,lat,lon


we renamed the columns name of the airports data (airport -> origin) inorder to match the column name in the dataframe
and made a left join to keep all rows from out dataframe, and only add matching airport info (latitude and longitude) when it exists.

In [501]:
airport_cord.columns = ["origin", "origin_lat", "origin_lng"]
df=pd.merge(df,airport_cord,on='origin', how='left')

we renamed the columns name of the airports data (airport -> dest) inorder to match the column name in the dataframe
and made a left join to keep all rows from out dataframe, and only add matching airport info (latitude and longitude) when it exists.

In [504]:
airport_cord.columns = ["dest", "dest_lat", "dest_lng"]
df=pd.merge(df,airport_cord,on='dest', how='left')

In [506]:
print(df['origin_lat'].isna().any())
print(df['origin_lng'].isna().any())
print(df['dest_lat'].isna().any())
print(df['dest_lng'].isna().any())

False
False
False
False


no missing values in origin_lat<br>
no missing values in origin_lng<br>
no missing values in dest_lat<br>
no missing values in dest_lng<br>
Every origin and dest in our dataframe matched with the airport_cord dataframe

In [509]:
df.head(3)

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng
0,2024,1,6,6,2024-01-06,475.0,BNA,"Nashville, TN",Tennessee,MKE,...,Wisconsin,WN,1404.0,1230,1405,0,36.124475,-86.678181,42.946932,-87.897064
1,2024,8,4,7,2024-08-04,2475.0,LAX,"Los Angeles, CA",California,JFK,...,New York,DL,951.0,1100,1957,0,33.942496,-118.408049,40.639928,-73.778692
2,2024,1,12,5,2024-01-12,296.0,SJC,"San Jose, CA",California,BUR,...,California,WN,2485.0,1035,1145,0,37.362995,-121.928621,34.200694,-118.358667


# Preprocessing and Cleaning the data

In [512]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 21 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   year               1000000 non-null  int64  
 1   month              1000000 non-null  int64  
 2   day_of_month       1000000 non-null  int64  
 3   day_of_week        1000000 non-null  int64  
 4   fl_date            1000000 non-null  str    
 5   distance           1000000 non-null  float64
 6   origin             1000000 non-null  str    
 7   origin_city_name   1000000 non-null  str    
 8   origin_state_nm    1000000 non-null  str    
 9   dest               1000000 non-null  str    
 10  dest_city_name     1000000 non-null  str    
 11  dest_state_nm      1000000 non-null  str    
 12  op_unique_carrier  1000000 non-null  str    
 13  op_carrier_fl_num  1000000 non-null  float64
 14  crs_dep_time       1000000 non-null  int64  
 15  crs_arr_time       1000000 non-null  int64  

## 1. Validate data types

In [515]:
df['fl_date']

0         2024-01-06
1         2024-08-04
2         2024-01-12
3         2024-04-03
4         2024-11-17
             ...    
999995    2024-11-10
999996    2024-05-27
999997    2024-12-12
999998    2024-05-20
999999    2024-07-24
Name: fl_date, Length: 1000000, dtype: str

We converted the fl_date column to datetime type

In [518]:
df['fl_date'] = pd.to_datetime(df['fl_date'])
df

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng
0,2024,1,6,6,2024-01-06,475.0,BNA,"Nashville, TN",Tennessee,MKE,...,Wisconsin,WN,1404.0,1230,1405,0,36.124475,-86.678181,42.946932,-87.897064
1,2024,8,4,7,2024-08-04,2475.0,LAX,"Los Angeles, CA",California,JFK,...,New York,DL,951.0,1100,1957,0,33.942496,-118.408049,40.639928,-73.778692
2,2024,1,12,5,2024-01-12,296.0,SJC,"San Jose, CA",California,BUR,...,California,WN,2485.0,1035,1145,0,37.362995,-121.928621,34.200694,-118.358667
3,2024,4,3,3,2024-04-03,2248.0,JFK,"New York, NY",New York,LAS,...,Nevada,B6,411.0,605,830,0,40.639928,-73.778692,36.080343,-115.152449
4,2024,11,17,7,2024-11-17,1824.0,CLE,"Cleveland, OH",Ohio,LAS,...,Nevada,WN,3473.0,1245,1420,0,41.409407,-81.854691,36.080343,-115.152449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,2024,11,10,7,2024-11-10,517.0,DEN,"Denver, CO",Colorado,SGU,...,Utah,OO,5051.0,2050,2245,0,39.861667,-104.673167,37.036378,-113.510303
999996,2024,5,27,1,2024-05-27,610.0,LGA,"New York, NY",New York,GSP,...,South Carolina,9E,5486.0,1820,2037,1,40.777242,-73.872606,34.895671,-82.218859
999997,2024,12,12,4,2024-12-12,386.0,SJC,"San Jose, CA",California,LAS,...,Nevada,NK,436.0,2044,2214,0,37.362995,-121.928621,36.080343,-115.152449
999998,2024,5,20,1,2024-05-20,837.0,MSY,"New Orleans, LA",Louisiana,ORD,...,Illinois,MQ,3888.0,600,833,0,29.993272,-90.259028,41.976940,-87.908150


In [520]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 21 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   year               1000000 non-null  int64         
 1   month              1000000 non-null  int64         
 2   day_of_month       1000000 non-null  int64         
 3   day_of_week        1000000 non-null  int64         
 4   fl_date            1000000 non-null  datetime64[us]
 5   distance           1000000 non-null  float64       
 6   origin             1000000 non-null  str           
 7   origin_city_name   1000000 non-null  str           
 8   origin_state_nm    1000000 non-null  str           
 9   dest               1000000 non-null  str           
 10  dest_city_name     1000000 non-null  str           
 11  dest_state_nm      1000000 non-null  str           
 12  op_unique_carrier  1000000 non-null  str           
 13  op_carrier_fl_num  1000000 non-null  fl

There is no meaning of the op_carrier_fl_num to be a float data, it is just a string id

In [523]:
df['op_carrier_fl_num']=df['op_carrier_fl_num'].astype(str)
df

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng
0,2024,1,6,6,2024-01-06,475.0,BNA,"Nashville, TN",Tennessee,MKE,...,Wisconsin,WN,1404.0,1230,1405,0,36.124475,-86.678181,42.946932,-87.897064
1,2024,8,4,7,2024-08-04,2475.0,LAX,"Los Angeles, CA",California,JFK,...,New York,DL,951.0,1100,1957,0,33.942496,-118.408049,40.639928,-73.778692
2,2024,1,12,5,2024-01-12,296.0,SJC,"San Jose, CA",California,BUR,...,California,WN,2485.0,1035,1145,0,37.362995,-121.928621,34.200694,-118.358667
3,2024,4,3,3,2024-04-03,2248.0,JFK,"New York, NY",New York,LAS,...,Nevada,B6,411.0,605,830,0,40.639928,-73.778692,36.080343,-115.152449
4,2024,11,17,7,2024-11-17,1824.0,CLE,"Cleveland, OH",Ohio,LAS,...,Nevada,WN,3473.0,1245,1420,0,41.409407,-81.854691,36.080343,-115.152449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,2024,11,10,7,2024-11-10,517.0,DEN,"Denver, CO",Colorado,SGU,...,Utah,OO,5051.0,2050,2245,0,39.861667,-104.673167,37.036378,-113.510303
999996,2024,5,27,1,2024-05-27,610.0,LGA,"New York, NY",New York,GSP,...,South Carolina,9E,5486.0,1820,2037,1,40.777242,-73.872606,34.895671,-82.218859
999997,2024,12,12,4,2024-12-12,386.0,SJC,"San Jose, CA",California,LAS,...,Nevada,NK,436.0,2044,2214,0,37.362995,-121.928621,36.080343,-115.152449
999998,2024,5,20,1,2024-05-20,837.0,MSY,"New Orleans, LA",Louisiana,ORD,...,Illinois,MQ,3888.0,600,833,0,29.993272,-90.259028,41.976940,-87.908150


In [524]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 21 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   year               1000000 non-null  int64         
 1   month              1000000 non-null  int64         
 2   day_of_month       1000000 non-null  int64         
 3   day_of_week        1000000 non-null  int64         
 4   fl_date            1000000 non-null  datetime64[us]
 5   distance           1000000 non-null  float64       
 6   origin             1000000 non-null  str           
 7   origin_city_name   1000000 non-null  str           
 8   origin_state_nm    1000000 non-null  str           
 9   dest               1000000 non-null  str           
 10  dest_city_name     1000000 non-null  str           
 11  dest_state_nm      1000000 non-null  str           
 12  op_unique_carrier  1000000 non-null  str           
 13  op_carrier_fl_num  1000000 non-null  st

## 2. Solving Inconsistency

Here we are checking the string columns, if they have an entry containins additional space (not trimmed string)

In [529]:
str_cols = ['origin', 'origin_city_name', 'origin_state_nm',
            'dest', 'dest_city_name', 'dest_state_nm',
            'op_unique_carrier', 'op_carrier_fl_num']

for c in str_cols:
    raw   = df[c].nunique()
    non_trimmed_strings = df[c].str.strip().str.upper().nunique()
    if raw == non_trimmed_strings:
        print(f"no issues at {c} ({raw} unique values)")
    else:
        print(f"{c}: {raw - non_trimmed_strings} values are not trimmed")

no issues at origin (348 unique values)
no issues at origin_city_name (342 unique values)
no issues at origin_state_nm (52 unique values)
no issues at dest (348 unique values)
no issues at dest_city_name (342 unique values)
no issues at dest_state_nm (52 unique values)
no issues at op_unique_carrier (15 unique values)
no issues at op_carrier_fl_num (6725 unique values)


Here we are checking the string columns, if they have an entry is repeated with other cases (such as: BNA -> bna, Bna)

In [531]:
str_cols = ['origin', 'origin_city_name', 'origin_state_nm',
            'dest', 'dest_city_name', 'dest_state_nm',
            'op_unique_carrier', 'op_carrier_fl_num']
for c in str_cols:
    raw   = df[c].nunique()
    capital_strings = df[c].str.upper().nunique() #this convert the string to capital then counts unique values
    if raw == capital_strings:
        print(f"No case problems at {c}, and ({raw} unique values)")
    else:
        print(f"There is a case problem at {c}, and ({raw-capital_strings} have problems)")

No case problems at origin, and (348 unique values)
No case problems at origin_city_name, and (342 unique values)
No case problems at origin_state_nm, and (52 unique values)
No case problems at dest, and (348 unique values)
No case problems at dest_city_name, and (342 unique values)
No case problems at dest_state_nm, and (52 unique values)
No case problems at op_unique_carrier, and (15 unique values)
No case problems at op_carrier_fl_num, and (6725 unique values)


In [534]:
(df["day_of_month"] != df["fl_date"].dt.day).any()

np.False_

In [536]:
df[df["day_of_week"] != df["fl_date"].dt.dayofweek]["day_of_week"].value_counts()

day_of_week
1    150977
5    149603
7    147383
4    147093
2    138160
3    137664
6    129120
Name: count, dtype: int64

In [538]:
(df["day_of_week"]-1 != df["fl_date"].dt.dayofweek).any()

np.False_

i wrote df["day_of_week"]-1 because the day_of_week in this dataset is encoded 1-7 instead of 0-6

In [541]:
(df["month"] != df["fl_date"].dt.month).any()

np.False_

In [543]:
(df["year"] != df["fl_date"].dt.year).any()

np.False_

In [545]:
#check if there is zero ot negative distances
df[df['distance']<=0]

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng


Here we check if there is an unusual arrival and departure time

In [548]:
df[(df['crs_dep_time'] > 2359) | (df['crs_dep_time'] < 0)]
df[(df['crs_arr_time'] > 2359) | (df['crs_arr_time'] < 0)]

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng


There are no unsual scheduled departure and arrival time

## 3. Duplicates

In [552]:
df[df.duplicated()]

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,dest_state_nm,op_unique_carrier,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng


No duplicates appeared in the dataframe

## 4. Missing data

In [555]:
df.isna().sum()

year                 0
month                0
day_of_month         0
day_of_week          0
fl_date              0
distance             0
origin               0
origin_city_name     0
origin_state_nm      0
dest                 0
dest_city_name       0
dest_state_nm        0
op_unique_carrier    0
op_carrier_fl_num    0
crs_dep_time         0
crs_arr_time         0
delayed              0
origin_lat           0
origin_lng           0
dest_lat             0
dest_lng             0
dtype: int64

As you see there are no Missing data in the dataframe

## 5. Outliers

In [560]:
df['distance_zscore']= (df['distance']-df['distance'].mean())/df['distance'].std()
df['distance_outlier'] = df['distance_zscore'].apply(lambda x: abs(x) >= 3)
df["distance_outlier"].value_counts()

distance_outlier
False    991429
True       8571
Name: count, dtype: int64

In [568]:
(df["distance_outlier"]==True).sum()/len(df["distance_outlier"])

np.float64(0.008571)

Only 0.86% of flights are distance outliers, that means the data is consistent.<br>
there exists some extremly short routes<br>
and exists some extremly long routes<br>

# Add features

Add a season column for the flights

In [573]:
seasons = []

for m in df['month']:
    if m <= 2:
        seasons.append('winter')
    elif m <= 5:
        seasons.append('spring')
    elif m <= 8:
        seasons.append('summer')
    else:
        seasons.append('fall')

df['season'] = seasons

A feature checking wheather the date is a weekend or not

In [576]:
df['is_weekend'] = df['day_of_week'] >= 6

In [578]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 25 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   year               1000000 non-null  int64         
 1   month              1000000 non-null  int64         
 2   day_of_month       1000000 non-null  int64         
 3   day_of_week        1000000 non-null  int64         
 4   fl_date            1000000 non-null  datetime64[us]
 5   distance           1000000 non-null  float64       
 6   origin             1000000 non-null  str           
 7   origin_city_name   1000000 non-null  str           
 8   origin_state_nm    1000000 non-null  str           
 9   dest               1000000 non-null  str           
 10  dest_city_name     1000000 non-null  str           
 11  dest_state_nm      1000000 non-null  str           
 12  op_unique_carrier  1000000 non-null  str           
 13  op_carrier_fl_num  1000000 non-null  st

We created the time_of_day feature ('afternoon','morning',...) 

In [581]:
df['departure_hour'] = df['crs_dep_time'] // 100

In [583]:
result = []

for h in df['departure_hour']:
    if 5 <= h <= 11:
        result.append('morning')
    elif 12 <= h <= 16:
        result.append('afternoon')
    elif 17 <= h <= 20:
        result.append('evening')
    else:
        result.append('night')

df['period_of_day'] = result

In [585]:
df.head(3)

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,season,is_weekend,departure_hour,period_of_day
0,2024,1,6,6,2024-01-06,475.0,BNA,"Nashville, TN",Tennessee,MKE,...,36.124475,-86.678181,42.946932,-87.897064,-0.601835,False,winter,True,12,afternoon
1,2024,8,4,7,2024-08-04,2475.0,LAX,"Los Angeles, CA",California,JFK,...,33.942496,-118.408049,40.639928,-73.778692,2.746703,False,summer,True,11,morning
2,2024,1,12,5,2024-01-12,296.0,SJC,"San Jose, CA",California,BUR,...,37.362995,-121.928621,34.200694,-118.358667,-0.901530,False,winter,False,10,morning


In [587]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 27 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   year               1000000 non-null  int64         
 1   month              1000000 non-null  int64         
 2   day_of_month       1000000 non-null  int64         
 3   day_of_week        1000000 non-null  int64         
 4   fl_date            1000000 non-null  datetime64[us]
 5   distance           1000000 non-null  float64       
 6   origin             1000000 non-null  str           
 7   origin_city_name   1000000 non-null  str           
 8   origin_state_nm    1000000 non-null  str           
 9   dest               1000000 non-null  str           
 10  dest_city_name     1000000 non-null  str           
 11  dest_state_nm      1000000 non-null  str           
 12  op_unique_carrier  1000000 non-null  str           
 13  op_carrier_fl_num  1000000 non-null  st

In [ ]:
df.to_csv('data/processed/processed_flights.csv')

In [2]:
import gdown 
import pandas as pd

### Load Dataset after Cleaing 

In [7]:

file_id = "1eztNdzA460wjWeqs-C-gF1L6tCI0a6ex"
url = f"https://drive.google.com/uc?id={file_id}"
output = "data/processed/flight_data_2024.csv"

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1eztNdzA460wjWeqs-C-gF1L6tCI0a6ex
From (redirected): https://drive.google.com/uc?id=1eztNdzA460wjWeqs-C-gF1L6tCI0a6ex&confirm=t&uuid=d66f78fd-f1cf-40ae-8e29-d557ec952f59
To: C:\Users\victus\OneDrive\Attachments\Desktop\Raheeq\Smstrs\8th_smstr_RaheeqMousa\Data science\Flight-Delay-Prediction\data\processed\flight_data_2024.csv
100%|██████████| 198M/198M [04:33<00:00, 724kB/s] 


'data/processed/flight_data_2024.csv'

### Read the dataset 

In [7]:
dataset = pd.read_csv('data/processed/flight_data_2024.csv')

### Review Dataset 

for this part we try to review the dataset after cleaning to make sure that there are no null value and we use .info() , isna() , head , tail 

In [9]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 28 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1000000 non-null  int64  
 1   year               1000000 non-null  int64  
 2   month              1000000 non-null  int64  
 3   day_of_month       1000000 non-null  int64  
 4   day_of_week        1000000 non-null  int64  
 5   fl_date            1000000 non-null  str    
 6   distance           1000000 non-null  float64
 7   origin             1000000 non-null  str    
 8   origin_city_name   1000000 non-null  str    
 9   origin_state_nm    1000000 non-null  str    
 10  dest               1000000 non-null  str    
 11  dest_city_name     1000000 non-null  str    
 12  dest_state_nm      1000000 non-null  str    
 13  op_unique_carrier  1000000 non-null  str    
 14  op_carrier_fl_num  1000000 non-null  int64  
 15  crs_dep_time       1000000 non-null  int64  

In [11]:
dataset.head()

,Unnamed: 0,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,...,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,season,is_weekend,departure_hour,period_of_day
0,0,2024,1,6,6,2024-01-06,475.0,BNA,"Nashville, TN",Tennessee,...,36.124475,-86.678181,42.946932,-87.897064,-0.601835,0,winter,1,12,afternoon
1,1,2024,8,4,7,2024-08-04,2475.0,LAX,"Los Angeles, CA",California,...,33.942496,-118.408049,40.639928,-73.778692,2.746703,0,summer,1,11,morning
2,2,2024,1,12,5,2024-01-12,296.0,SJC,"San Jose, CA",California,...,37.362995,-121.928621,34.200694,-118.358667,-0.901530,0,winter,0,10,morning
3,3,2024,4,3,3,2024-04-03,2248.0,JFK,"New York, NY",New York,...,40.639928,-73.778692,36.080343,-115.152449,2.366644,0,spring,0,6,morning
4,4,2024,11,17,7,2024-11-17,1824.0,CLE,"Cleveland, OH",Ohio,...,41.409407,-81.854691,36.080343,-115.152449,1.656754,0,fall,1,12,afternoon


In [13]:
dataset.tail()

,Unnamed: 0,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,...,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,season,is_weekend,departure_hour,period_of_day
999995,999995,2024,11,10,7,2024-11-10,517.0,DEN,"Denver, CO",Colorado,...,39.861667,-104.673167,37.036378,-113.510303,-0.531516,0,fall,1,20,evening
999996,999996,2024,5,27,1,2024-05-27,610.0,LGA,"New York, NY",New York,...,40.777242,-73.872606,34.895671,-82.218859,-0.375809,0,spring,0,18,evening
999997,999997,2024,12,12,4,2024-12-12,386.0,SJC,"San Jose, CA",California,...,37.362995,-121.928621,36.080343,-115.152449,-0.750845,0,fall,0,20,evening
999998,999998,2024,5,20,1,2024-05-20,837.0,MSY,"New Orleans, LA",Louisiana,...,29.993272,-90.259028,41.976940,-87.908150,0.004250,0,spring,0,6,morning
999999,999999,2024,7,24,3,2024-07-24,1303.0,SAN,"San Diego, CA",California,...,32.733563,-117.189663,29.984435,-95.341442,0.784459,0,summer,0,13,afternoon


In [15]:
dataset.columns

Index(['Unnamed: 0', 'year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
       'distance', 'origin', 'origin_city_name', 'origin_state_nm', 'dest',
       'dest_city_name', 'dest_state_nm', 'op_unique_carrier',
       'op_carrier_fl_num', 'crs_dep_time', 'crs_arr_time', 'delayed',
       'origin_lat', 'origin_lng', 'dest_lat', 'dest_lng', 'distance_zscore',
       'distance_outlier', 'season', 'is_weekend', 'departure_hour',
       'period_of_day'],
      dtype='str')

In [17]:
dataset.isna().sum()

Unnamed: 0           0
year                 0
month                0
day_of_month         0
day_of_week          0
fl_date              0
distance             0
origin               0
origin_city_name     0
origin_state_nm      0
dest                 0
dest_city_name       0
dest_state_nm        0
op_unique_carrier    0
op_carrier_fl_num    0
crs_dep_time         0
crs_arr_time         0
delayed              0
origin_lat           0
origin_lng           0
dest_lat             0
dest_lng             0
distance_zscore      0
distance_outlier     0
season               0
is_weekend           0
departure_hour       0
period_of_day        0
dtype: int64

### Drop unnamed column 

In [20]:
dataset = dataset.drop(columns=['Unnamed: 0'])

### Review the count for delayed and not delayed flights 

we can see that delayed flight it's about 20% of the dataset and not delayed flights it's about 70% of the dataset 

In [33]:
dataset['delayed'].value_counts()

delayed
0    791829
1    208171
Name: count, dtype: int64

### Summarization , interpretation 

In [36]:
dataset.mode()

,year,month,day_of_month,day_of_week,fl_date,distance,origin,origin_city_name,origin_state_nm,dest,...,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,season,is_weekend,departure_hour,period_of_day
0,2024,7,15,1,2024-10-17,337.0,ATL,"Chicago, IL",Texas,ATL,...,33.6367,-84.427864,33.6367,-84.427864,-0.832885,0,fall,0,7,morning


### Interpretation of Mode 
The mode represents the most frequently occurring value in each column

### Categorical Data Analysis

## Cheack which season is the most season that may have delayed 

In [43]:
dataset[dataset['delayed'] == 1]["season"].value_counts()

season
summer    68168
spring    56217
fall      53836
winter    29950
Name: count, dtype: int64

## find the persantage of delay for each season 

In [46]:
dataset.groupby("season")["delayed"].mean().sort_values()

season
fall      0.160014
winter    0.200977
spring    0.221921
summer    0.260967
Name: delayed, dtype: float64

In [48]:
dataset[dataset['delayed'] == 1]["period_of_day"].value_counts()

period_of_day
afternoon    67666
evening      65055
morning      57363
night        18087
Name: count, dtype: int64

In [50]:
dataset.groupby("period_of_day")["delayed"].mean().sort_values()

period_of_day
morning      0.137376
afternoon    0.231360
night        0.268992
evening      0.292084
Name: delayed, dtype: float64

In [52]:
dataset[dataset['delayed'] == 1]["dest_city_name"].value_counts()

dest_city_name
Chicago, IL                       10515
Dallas/Fort Worth, TX             10470
Atlanta, GA                        8410
New York, NY                       8133
Denver, CO                         8077
                                  ...  
Moab, UT                              1
New Bern/Morehead/Beaufort, NC        1
Pago Pago, TT                         1
Dillingham, AK                        1
Morgantown, WV                        1
Name: count, Length: 338, dtype: int64

In [54]:
dataset[dataset['delayed'] == 1]["month"].value_counts()

month
7     26093
5     22924
6     21729
8     20346
1     18299
12    17625
3     17524
4     15769
9     12814
11    11927
2     11651
10    11470
Name: count, dtype: int64

In [56]:
dataset[dataset['delayed'] == 1]["day_of_month"].value_counts()

day_of_month
22    7700
15    7638
16    7378
19    7359
9     7310
21    7281
27    7159
8     7153
18    7150
28    7120
26    7067
29    6999
23    6976
17    6963
20    6907
12    6837
7     6789
24    6732
11    6710
5     6667
6     6600
14    6570
10    6503
2     6496
3     6451
25    6345
4     6268
30    5945
13    5898
1     5842
31    3358
Name: count, dtype: int64

## interpration Categorical Data Analysis
- we can see that summer is the season with the highest number of delays. Next, we have spring, followed closely by fall, which shows very similar results.
- we can see that afternoon is the period_of_day with the highest number of delays , then we have evening is very close to it  which it mean the most delayed cases happen on the afternoon and evening 
- we see that Chicago, IL  is the city that have the most delayed 

### Numerical Data Analysis

## Feature Correlation Analysis

In [65]:
dataset.corr(numeric_only=True)

,year,month,day_of_month,day_of_week,distance,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,is_weekend,departure_hour
year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month,NaN,1.000000,0.003272,0.006569,-0.004969,0.031510,-0.003422,-0.008256,-0.038527,0.019580,-0.004784,0.016910,-0.006033,-0.004969,-0.002339,0.012381,-0.003318
day_of_month,NaN,0.003272,1.000000,0.012904,0.001442,0.003405,0.001653,-0.000545,0.009072,-0.000169,0.000835,-0.001001,-0.000486,0.001442,-0.001175,0.014172,0.001650
day_of_week,NaN,0.006569,0.012904,1.000000,0.008157,-0.005467,0.003731,-0.000665,0.027621,-0.008472,-0.002898,-0.008459,-0.002243,0.008157,0.001031,0.784152,0.003677
distance,NaN,-0.004969,0.001442,0.008157,1.000000,-0.372124,-0.008622,0.015483,0.016279,-0.047730,-0.115411,-0.048448,-0.116084,1.000000,0.344296,0.011612,-0.008828
op_carrier_fl_num,NaN,0.031510,0.003405,-0.005467,-0.372124,1.000000,0.027906,0.017634,-0.017495,0.093442,0.129872,0.098733,0.130769,-0.372124,-0.092020,-0.006005,0.027571
crs_dep_time,NaN,-0.003422,0.001653,0.003731,-0.008622,0.027906,1.000000,0.685170,0.161730,-0.031345,0.000153,0.031230,-0.007995,-0.008622,0.015623,-0.000825,0.999331
crs_arr_time,NaN,-0.008256,-0.000545,-0.000665,0.015483,0.017634,0.685170,1.000000,0.135774,-0.013375,0.015071,0.017287,0.004897,0.015483,-0.018303,-0.002098,0.684257
delayed,NaN,-0.038527,0.009072,0.027621,0.016279,-0.017495,0.161730,0.135774,1.000000,-0.014733,0.027353,-0.010078,0.015965,0.016279,-0.001716,0.011146,0.161462
origin_lat,NaN,0.019580,-0.000169,-0.008472,-0.047730,0.093442,-0.031345,-0.013375,-0.014733,1.000000,0.024407,0.229931,-0.016001,-0.047730,-0.045395,-0.011965,-0.031945


### Interpretation of Correlation Results

for this interpretaion i try to find the most strong and weak relationships 

- very strong relationship between departure_hour and crs_dep_time (0.9993), which is expected since departure_hour was directly bulid  from crs_dep_time. 
-  crs_dep_time and crs_arr_time share a moderate relationship (0.685), as flights departing later also tend to arrive later. 
- strong relationship between is_weekend and day_of_week (0.784), consistent with how the feature was constructed. 
-  moderate relationship between origin_lng and dest_lng (0.604), and origin_lat and dest_lat (0.230), reflecting that many flights operate within similar geographic corridors. 
- The distance and distance_zscore columns are perfectly correlated (1.000) since the z-score is derived directly from distance. 
- The distance_outlier and distance_zscore show a moderate positive relationship (0.344), as expected. 
- The year column produces NaN values across all pairs due to zero variance (all records are from 2024).

In [68]:
dataset.describe()

,year,month,day_of_month,day_of_week,distance,op_carrier_fl_num,crs_dep_time,crs_arr_time,delayed,origin_lat,origin_lng,dest_lat,dest_lng,distance_zscore,distance_outlier,is_weekend,departure_hour
count,1000000.0,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1.000000e+06,1000000.000000,1000000.000000,1000000.000000
mean,2024.0,6.597970,15.786584,3.983077,834.461610,2502.772380,1325.796484,1490.292545,0.208171,36.597999,-94.832342,36.596790,-94.843876,9.974599e-17,0.008571,0.276503,12.988599
std,0.0,3.396064,8.787083,2.010310,597.275568,1652.043715,492.424283,518.698428,0.406000,5.987153,18.546931,5.994319,18.559162,1.000000e+00,0.092182,0.447269,4.894416
min,2024.0,1.000000,1.000000,1.000000,31.000000,1.000000,1.000000,1.000000,0.000000,-14.331662,-176.642482,-14.331662,-176.642482,-1.345211e+00,0.000000,0.000000,0.000000
25%,2024.0,4.000000,8.000000,2.000000,399.000000,1150.000000,905.000000,1103.000000,0.000000,32.898639,-111.150260,32.898639,-111.150260,-7.290799e-01,0.000000,0.000000,9.000000
50%,2024.0,7.000000,16.000000,4.000000,680.000000,2224.000000,1320.000000,1515.000000,0.000000,36.894604,-87.908150,36.894604,-87.908150,-2.586103e-01,0.000000,0.000000,13.000000
75%,2024.0,10.000000,23.000000,6.000000,1069.000000,3717.000000,1735.000000,1925.000000,0.000000,40.777242,-80.951379,40.777242,-80.951379,3.926804e-01,0.000000,1.000000,17.000000
max,2024.0,12.000000,31.000000,7.000000,5095.000000,8819.000000,2359.000000,2359.000000,1.000000,71.284861,145.729986,71.284861,145.729986,7.133288e+00,1.000000,1.000000,23.000000


### Interpretation of describe Results

- All the dataset came form 2024 year and it's take the whole month and all days on the month and we can see that by the mean of the month (6.5) and day_of_month (15) and all the days on the week (3) 
- The average flight distance is approximately 834 miles, with a minimum of 31 miles and a maximum of 5,095 miles, indicating a wide range of short-haul to long-haul domestic flights. 
- Scheduled departure times (crs_dep_time) range from 0001 to 2359 with a mean around 1326 (early afternoon), while arrival times average around 1490.
- The target variable delayed has a mean of 0.208, confirming the class imbalance where approximately 20.8% of flights are delayed. 
- Latitude values for origin airports range from −14.3 to 71.3, and longitude from −176.6 to 145.7, reflecting the geographic diversity of U.S. domestic airports including Alaska, Hawaii, and U.S. territories.

In [71]:
dataset.corr(numeric_only=True)["delayed"]

year                      NaN
month               -0.038527
day_of_month         0.009072
day_of_week          0.027621
distance             0.016279
op_carrier_fl_num   -0.017495
crs_dep_time         0.161730
crs_arr_time         0.135774
delayed              1.000000
origin_lat          -0.014733
origin_lng           0.027353
dest_lat            -0.010078
dest_lng             0.015965
distance_zscore      0.016279
distance_outlier    -0.001716
is_weekend           0.011146
departure_hour       0.161462
Name: delayed, dtype: float64

### Interpretation of Correlation Results relates to delayed 
We can see that most columns have almost no relationship with delayed. However, crs_dep_time, crs_arr_time, and departure_hour show a weak positive relationship with delayed, with correlation values of 0.161730, 0.135774, and 0.161462, respectively.

### Overall interpreation for the numerical Data Analysis

- The low correlation values confirm that flight delay prediction is a non-linear problem, which justifies the use of tree-based ensemble models such as Random Forest and XGBoost.